In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# --- SCHRITT 1: Daten laden.
df = pd.read_csv("Datasets/digits.csv")  # Pfad ggf. anpassen
y = df["class"]
X = df.drop(columns=["class"])

In [ ]:
# --- Schritt 1.1: EDA (explorative Datenanalyse)
import matplotlib.pyplot as plt

# Tabellarische Ansicht der Daten
print(df.head(3))

# beschreibende Statistik
print(df.describe())
# Verteilung der Klassen
print(y.value_counts().sort_index())

In [ ]:
# --- SCHRITT 2: Der "Goldene Schnitt" (Train-Test-Split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# --- SCHRITT 3: Die Pipeline bauen ---
pipeline = Pipeline([
    ('scaler',      StandardScaler()),
    ('pca',         PCA(n_components=0.95, random_state=42)),
    ('classifier',  RandomForestClassifier(random_state=42, n_estimators=100))
])

In [ ]:
# --- SCHRITT 4: K-Fold Cross Validation (Training & Validierung)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
 
scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
print("\n--- Cross-Validation Ergebnisse ---")
print(f"Genauigkeit in den Folds: {scores}")
print(f"Durchschnittliche Genauigkeit: {np.mean(scores):.2%} (+/- {np.std(scores):.2%})")

In [ ]:
# --- SCHRITT 5: Das finale Modell trainieren ---
pipeline.fit(X_train, y_train)

In [ ]:
# --- SCHRITT 6: Finale Evaluation (Die Stunde der Wahrheit) ---
final_score = pipeline.score(X_test, y_test)
print(f"\nFinale Genauigkeit auf dem Test-Set: {final_score:.2%}")

In [ ]:
# --- SCHRITT 7: Modell speichern (Deployment) ---
joblib.dump(pipeline, 'Modelle/digits_random_forest.pkl')
print("\nModell wurde gespeichert.")